In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt

import os
from pathlib import Path

from vivarium_helpers.utils import convert_to_categorical, constant_categorical, print_memory_usage, Timer

from vivarium_helpers.projects.alzheimers import loading, population
from vivarium_helpers.projects.alzheimers.population import RunType
from vivarium_helpers.projects.alzheimers.results import AlzheimersResultsProcessor
from vivarium_helpers.utils import print_memory_usage, convert_to_categorical

!date
!whoami
!pwd

Thu Mar  5 15:02:19 PST 2026
ndbs
/mnt/share/code/ndbs/vivarium_research_alzheimers/results_tables


In [15]:
from vivarium_helpers.vph_output.loading import VPHOutput

In [4]:
results_dir = Path("/snfs1/Project/simulation_science/alzheimers/results_in_progress")
!ls $results_dir

bbbm_tests.csv	   deaths.csv	   medication_doses.csv  version1  version4
csf_pet_tests.csv  incidence.csv   population.csv	 version2  version5
dalys.csv	   medication.csv  prevalence.csv	 version3


In [18]:
with Timer():
    output = VPHOutput.from_directory(results_dir, '.csv')
    print(output.table_names())

for key in output:
    print_memory_usage(output[key], f'{key} (raw)')
    output[key] = output[key].pipe(convert_to_categorical)
    print_memory_usage(output[key], key)

['deaths', 'incidence', 'dalys', 'medication', 'population', 'prevalence', 'csf_pet_tests', 'bbbm_tests', 'medication_doses']
Elapsed time: 0:00:19.680741
245.259732 MB deaths (raw)
79.350659 MB deaths
429.647892 MB incidence (raw)
142.825894 MB incidence
736.462932 MB dalys (raw)
238.038798 MB dalys
52.554436 MB medication (raw)
16.932279 MB medication
64.665492 MB population (raw)
23.706917 MB population
729.075732 MB prevalence (raw)
238.038787 MB prevalence
480.943332 MB csf_pet_tests (raw)
158.694833 MB csf_pet_tests
103.21178 MB bbbm_tests (raw)
33.858977 MB bbbm_tests
122.433396 MB medication_doses (raw)
38.255 MB medication_doses


In [5]:
bbbm_tests = pd.read_csv(results_dir / "bbbm_tests.csv")
print_memory_usage(bbbm_tests, "bbbm_tests")
prevalence = pd.read_csv(results_dir / "prevalence.csv")
print_memory_usage(prevalence, "prevalence")

103.21178 MB bbbm_tests
729.075732 MB prevalence


In [6]:
bbbm_tests = convert_to_categorical(bbbm_tests)
print_memory_usage(bbbm_tests, "bbbm_tests")
prevalence = convert_to_categorical(prevalence)
print_memory_usage(prevalence, "prevalence")

33.858977 MB bbbm_tests
238.038787 MB prevalence


In [7]:
person_time = pd.read_csv(results_dir / "population.csv").pipe(convert_to_categorical)
print_memory_usage(person_time, "population")

23.706917 MB population


In [8]:
bbbm_tests.head()

,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Mean,95% UI Lower,...,draw_187,draw_199,draw_204,draw_211,draw_219,draw_235,draw_236,draw_245,draw_248,draw_249
0,2025,Brazil,65_to_69,Female,Preclinical AD,BBBM Testing Only,BBBM Tests,Number,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025,Brazil,65_to_69,Male,Preclinical AD,BBBM Testing Only,BBBM Tests,Number,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025,Brazil,65_to_69,Both,Preclinical AD,BBBM Testing Only,BBBM Tests,Number,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2025,Brazil,65_to_69,Female,Preclinical AD,BBBM Testing and Treatment,BBBM Tests,Number,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025,Brazil,65_to_69,Male,Preclinical AD,BBBM Testing and Treatment,BBBM Tests,Number,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Look up BBBM tests in the US in 2065

In [9]:
bbbm_tests.Measure.unique()

['BBBM Tests', 'Positive BBBM Tests']
Categories (2, object): ['BBBM Tests', 'Positive BBBM Tests']

In [32]:
df_tests = bbbm_tests.query(
    "Location == 'United States of America' and Year == 2065"
    " and Sex == 'Both' and Age == '65_plus'"
    " and Metric == 'Number'"
    " and Scenario == 'BBBM Testing Only'" # Treatment scenario has the same values
).set_index(['Disease Stage', 'Measure'])
df_tests

Year                  Location      Age  \
Disease Stage  Measure                                                        
Preclinical AD BBBM Tests           2065  United States of America  65_plus   
               Positive BBBM Tests  2065  United States of America  65_plus   
Susceptible    BBBM Tests           2065  United States of America  65_plus   
               Positive BBBM Tests  2065  United States of America  65_plus   

                                     Sex           Scenario  Metric  \
Disease Stage  Measure                                                
Preclinical AD BBBM Tests           Both  BBBM Testing Only  Number   
               Positive BBBM Tests  Both  BBBM Testing Only  Number   
Susceptible    BBBM Tests           Both  BBBM Testing Only  Number   
               Positive BBBM Tests  Both  BBBM Testing Only  Number   

                                            Mean  95% UI Lower  95% UI Upper  \
Disease Stage  Measure                                                         
Preclinical AD BBBM Tests           3.885017e+05  3.491080e+05  4.355455e+05   
               Positive BBBM Tests  1.942491e+05  1.751445e+05  2.176239e+05   
Susceptible    BBBM Tests           9.254922e+06  8.419039e+06  9.874273e+06   
               Positive BBBM Tests  1.850984e+04  1.683808e+04  1.974855e+04   

                                          draw_1  ...      draw_187  \
Disease Stage  Measure                            ...                 
Preclinical AD BBBM Tests           3.915858e+05  ...  4.148537e+05   
               Positive BBBM Tests  1.954958e+05  ...  2.073191e+05   
Susceptible    BBBM Tests           9.742138e+06  ...  9.367058e+06   
               Positive BBBM Tests  1.948428e+04  ...  1.873412e+04   

                                        draw_199      draw_204      draw_211  \
Disease Stage  Measure                                                         
Preclinical AD BBBM Tests           4.305728e+05  4.375816e+05  3.881469e+05   
               Positive BBBM Tests  2.140334e+05  2.188444e+05  1.943546e+05   
Susceptible    BBBM Tests           9.528150e+06  1.007248e+07  9.086001e+06   
               Positive BBBM Tests  1.905630e+04  2.014495e+04  1.817200e+04   

                                        draw_219      draw_235      draw_236  \
Disease Stage  Measure                                                         
Preclinical AD BBBM Tests           3.998631e+05  3.998077e+05  3.767044e+05   
               Positive BBBM Tests  1.996768e+05  1.999440e+05  1.882756e+05   
Susceptible    BBBM Tests           8.826405e+06  9.264114e+06  8.602443e+06   
               Positive BBBM Tests  1.765281e+04  1.852823e+04  1.720489e+04   

                                        draw_245      draw_248      draw_249  
Disease Stage  Measure                                                        
Preclinical AD BBBM Tests           4.103154e+05  3.960462e+05  3.827623e+05  
               Positive BBBM Tests  2.044067e+05  1.967630e+05  1.916626e+05  
Susceptible    BBBM Tests           9.234202e+06  9.441887e+06  8.956886e+06  
               Positive BBBM Tests  1.846840e+04  1.888377e+04  1.791377e+04  

[4 rows x 34 columns]

In [33]:
df_tests.loc[(slice(None), 'BBBM Tests'), 'Mean'].sum()

9643423.195666172

In [ ]:
# Total tests = BBBM Tests summed across Preclinical and Susceptible
print(total_tests := 9.254922e+06 + 3.885017e+05, "Total tests")
df_tests.loc[(slice(None), 'BBBM Tests'), 'Mean'].sum()

# Fraction of tests among susceptible population = (BBBM Tests among Susceptible) / (Total tests)
print(fraction_preclinical_tests := 9.254922e+06 / total_tests, "Fraction of tests among susceptible population")

# False positive tests = positive tests among susceptible population
print(1.850984e+04, "False positive tests")

# False negative tests = (total tests) - (positive tests) among preclinical population
print(3.885017e+05 - 1.942491e+05, "False negative tests")

9643423.7 Total tests
0.9597133018224638 Fraction of tests among susceptible population
18509.84 False positive tests
194252.6 False negative tests


# Investigate missing values

Rates for BBBM tests were all NaN for Susceptible population in intervention scenarios

In [ ]:
# Rates are all NaN for Susceptible population in intervention scenarios
bbbm_tests.query(
    "Mean.isna()"
)[['Age', 'Disease Stage', 'Scenario', 'Metric']].value_counts()

Age       Disease Stage  Scenario                    Metric
65_plus   Susceptible    BBBM Testing Only           Rate      4560
                         BBBM Testing and Treatment  Rate      4560
65_to_69  Susceptible    BBBM Testing Only           Rate      4560
                         BBBM Testing and Treatment  Rate      4560
70_to_74  Susceptible    BBBM Testing Only           Rate      4560
                         BBBM Testing and Treatment  Rate      4560
75_to_79  Susceptible    BBBM Testing Only           Rate      4560
                         BBBM Testing and Treatment  Rate      4560
dtype: int64

In [12]:
person_time

,Year,Location,Age,Sex,Scenario,Measure,Metric,Mean,95% UI Lower,95% UI Upper,...,draw_187,draw_199,draw_204,draw_211,draw_219,draw_235,draw_236,draw_245,draw_248,draw_249
0,2025,United States of America,65_to_69,Female,Reference,Population,Number,1.053025e+07,9.602182e+06,1.119637e+07,...,1.066321e+07,1.087770e+07,1.142234e+07,1.026845e+07,1.003314e+07,1.052835e+07,9.845692e+06,1.054023e+07,1.080614e+07,1.012157e+07
1,2025,United States of America,65_to_69,Male,Reference,Population,Number,9.439692e+06,8.612104e+06,1.003853e+07,...,9.557105e+06,9.760703e+06,1.023595e+07,9.194452e+06,8.998697e+06,9.438117e+06,8.832245e+06,9.451196e+06,9.690784e+06,9.076959e+06
2,2025,United States of America,65_to_69,Both,Reference,Population,Number,1.996994e+07,1.821429e+07,2.123490e+07,...,2.022031e+07,2.063840e+07,2.165829e+07,1.946290e+07,1.903184e+07,1.996647e+07,1.867794e+07,1.999142e+07,2.049692e+07,1.919853e+07
3,2025,United States of America,65_to_69,Female,BBBM Testing Only,Population,Number,1.053025e+07,9.602182e+06,1.119637e+07,...,1.066321e+07,1.087770e+07,1.142234e+07,1.026845e+07,1.003314e+07,1.052835e+07,9.845692e+06,1.054023e+07,1.080614e+07,1.012157e+07
4,2025,United States of America,65_to_69,Male,BBBM Testing Only,Population,Number,9.439692e+06,8.612104e+06,1.003853e+07,...,9.557105e+06,9.760703e+06,1.023595e+07,9.194452e+06,8.998697e+06,9.438117e+06,8.832245e+06,9.451196e+06,9.690784e+06,9.076959e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102595,2100,United Kingdom,65_plus,Male,BBBM Testing Only,Population,Number,8.211104e+06,7.812346e+06,8.586794e+06,...,8.155819e+06,8.163366e+06,8.298154e+06,8.198380e+06,8.184304e+06,8.097882e+06,7.963680e+06,8.143412e+06,7.909882e+06,8.554832e+06
102596,2100,United Kingdom,65_plus,Both,BBBM Testing Only,Population,Number,1.781940e+07,1.685420e+07,1.865542e+07,...,1.768271e+07,1.773710e+07,1.803271e+07,1.789857e+07,1.784744e+07,1.746192e+07,1.735553e+07,1.764959e+07,1.721110e+07,1.859581e+07
102597,2100,United Kingdom,65_plus,Female,BBBM Testing and Treatment,Population,Number,9.624290e+06,9.065585e+06,1.008647e+07,...,9.541777e+06,9.591930e+06,9.748152e+06,9.715561e+06,9.678139e+06,9.378973e+06,9.406334e+06,9.522937e+06,9.318131e+06,1.006098e+07
102598,2100,United Kingdom,65_plus,Male,BBBM Testing and Treatment,Population,Number,8.216580e+06,7.816583e+06,8.592453e+06,...,8.160491e+06,8.169314e+06,8.302163e+06,8.203563e+06,8.189536e+06,8.103463e+06,7.969092e+06,8.149951e+06,7.915511e+06,8.561145e+06


In [14]:
bbbm_tests5 = pd.read_csv(results_dir / "version5/bbbm_tests.csv").pipe(convert_to_categorical)
bbbm_tests5.isna().sum()

Year             0
Location         0
Age              0
Sex              0
Disease Stage    0
Scenario         0
Measure          0
Metric           0
Mean             0
95% UI Lower     0
95% UI Upper     0
dtype: int64

# Write a function to verify output

In [ ]:
def verify_output(df):
    assert not df.isna().any().any(), "DataFrame contains missing values"
    
    

In [25]:
for key, df in output.items():
    print(key)
    print(df.Age.unique())

deaths
['65_to_69', '70_to_74', '75_to_79', '80_to_84', '85_to_89', ..., '45_to_49', '50_to_54', '55_to_59', '60_to_64', '65_plus']
Length: 15
Categories (15, object): ['30_to_34', '35_to_39', '40_to_44', '45_to_49', ..., '80_to_84', '85_to_89', '90_to_94', '95_plus']
incidence
['65_to_69', '70_to_74', '75_to_79', '80_to_84', '85_to_89', ..., '45_to_49', '50_to_54', '55_to_59', '60_to_64', '65_plus']
Length: 15
Categories (15, object): ['30_to_34', '35_to_39', '40_to_44', '45_to_49', ..., '80_to_84', '85_to_89', '90_to_94', '95_plus']
dalys
['65_to_69', '70_to_74', '75_to_79', '80_to_84', '85_to_89', ..., '45_to_49', '50_to_54', '55_to_59', '60_to_64', '65_plus']
Length: 15
Categories (15, object): ['30_to_34', '35_to_39', '40_to_44', '45_to_49', ..., '80_to_84', '85_to_89', '90_to_94', '95_plus']
medication
['65_to_69', '70_to_74', '75_to_79', '65_plus']
Categories (4, object): ['65_plus', '65_to_69', '70_to_74', '75_to_79']
population
['65_to_69', '70_to_74', '75_to_79', '80_to_84', 

In [ ]:
for key, df in output.items():
    print(key)
    try:
        verify_output(df)
    except(AssertionError) as e:
        print(e)

deaths
incidence
dalys
medication
DataFrame contains missing values
population
prevalence
csf_pet_tests
bbbm_tests
DataFrame contains missing values
medication_doses


# Try calculating increased time in preclinical and/or decreased time in dementia